In [ ]:
import os

# policy
from huggingface_hub import snapshot_download
from lerobot.configs.policies import PreTrainedConfig

# record utils
from lerobot.record import record, RecordConfig, DatasetRecordConfig

# torch
from torch import cuda

# utils
from dotenv import load_dotenv

# my code
from src.robot_config import robot_config
from src.paths import REPO_ROOT, HF_NAME, POLICIES_DIR, EVAL_DIR
from src.robot_const import TABLE_START_POSE_OPEN, FOLDED_START_POSE
from src.utils import check_resume

# set up env secrets
load_dotenv(REPO_ROOT / ".env", override=True)

# cuda
device = "cuda" if cuda.is_available() else "cpu"
print(f"Using device: {device}")

%load_ext autoreload
%autoreload 2

Using device: cuda
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


Set Params:

In [8]:
PUSH_TO_HUB       = False
SAVE_TO_DATASET   = True
REPO_NAME         = 'so101_leg_pick_and_place'
EXPERIMENT_NAME   = '50_episodes_v2'
POLICY_CHECKPOINT = '100000'
POLICY_TYPE       = 'act'
NUM_EPISODES      = 5
FPS               = 30
EPISODE_TIME_SEC  = 60
RESET_TIME_SEC    = 5
EVAL_TYPE         = 'real_v1'

Log to HF if pushing:

In [9]:
if PUSH_TO_HUB:
    os.system(f"huggingface-cli login --token {os.getenv('HUGGINGFACE_TOKEN')}")

Initialize Policy:

In [10]:
# Resolve path or HF id
policy_path = POLICIES_DIR / POLICY_TYPE / REPO_NAME / EXPERIMENT_NAME / "checkpoints" / POLICY_CHECKPOINT / "pretrained_model"

if not policy_path.exists():
    # load from Hugging Face Hub
    policy_id = f"{HF_NAME}/{POLICY_TYPE}-{REPO_NAME}-{EXPERIMENT_NAME}"
    snapshot_download(
    repo_id                = policy_id,
    revision               = "main",
    local_dir              = str(policy_path),
    local_dir_use_symlinks = False
    )

policy_config = PreTrainedConfig.from_pretrained(policy_path)
policy_config.pretrained_path = policy_path # type: ignore

/home/jonathan/miniforge3/envs/lerobot-sim-real-sim/lib/python3.10/site-packages/huggingface_hub/file_download.py:982: UserWarning: `local_dir_use_symlinks` parameter is deprecated and will be ignored. The process to download files to a local folder has been updated and do not rely on symlinks anymore. You only need to pass a destination folder as`local_dir`.
For more details, check out https://huggingface.co/docs/huggingface_hub/main/en/guides/download#download-files-to-local-folder.
  warnings.warn(
Fetching 8 files: 100%|██████████| 8/8 [00:08<00:00,  1.02s/it]


Build Dataset:

In [11]:
dataset_path = EVAL_DIR / POLICY_TYPE / REPO_NAME / f"{EXPERIMENT_NAME}-{EVAL_TYPE}"
resume = check_resume(dataset_path)

dscfg = DatasetRecordConfig(
    repo_id                             = f"{HF_NAME}/eval_{REPO_NAME}_{EXPERIMENT_NAME}_{EVAL_TYPE}",
    single_task                         = f"eval dataset for {REPO_NAME} with policy {EXPERIMENT_NAME}, mode = {EVAL_TYPE}",
    root                                = dataset_path.__str__(),
    fps                                 = FPS,
    episode_time_s                      = EPISODE_TIME_SEC,
    reset_time_s                        = RESET_TIME_SEC,
    num_episodes                        = NUM_EPISODES,
    video                               = True,
    push_to_hub                         = PUSH_TO_HUB,
    private                             = True,
    num_image_writer_processes          = 0,
    num_image_writer_threads_per_camera = 4,
    video_encoding_batch_size           = 1,
)

rc = RecordConfig(
    robot        = robot_config,
    dataset      = dscfg,
    teleop       = None,
    policy       = policy_config,
    display_data = True,
    play_sounds  = True,
    resume       = resume
)

In [12]:
record(rc, reset_pose = TABLE_START_POSE_OPEN, give_feedback = True)

Loading weights from local directory


INFO 2025-10-02 12:47:43 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-Sonix_Technology_Co.__Ltd._USB2.0_CAM1_USB2.0_CAM1-video-index0) connected.
INFO 2025-10-02 12:47:45 a_opencv.py:180 OpenCVCamera(/dev/v4l/by-id/usb-046d_Logitech_BRIO_8F54E371-video-index0) connected.
INFO 2025-10-02 12:47:45 follower.py:104 so_101_follower SO101Follower connected.
INFO 2025-10-02 12:47:47 ls/utils.py:227 Recording episode 0
INFO 2025-10-02 12:48:47 ls/utils.py:227 Reset the environment
INFO 2025-10-02 12:48:48 ls/utils.py:227 Input success
[2025-10-02T09:49:01Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
[2025-10-02T09:49:01Z WARN  wgpu_hal::vulkan::conv] Unrecognized present mode 1000361000
Map: 100%|██████████| 1729/1729 [00:00<00:00, 3169.74 examples/s]
Generating train split: 1729 examples [00:00, 683050.92 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 2025011

Escape key pressed. Stopping data recording...


Map: 100%|██████████| 1735/1735 [00:00<00:00, 3398.38 examples/s]
Generating train split: 3464 examples [00:00, 1202538.41 examples/s]
Svt[info]: -------------------------------------------
Svt[info]: SVT [version]:	SVT-AV1 Encoder Lib v3.0.0
Svt[info]: SVT [build]  :	GCC 14.2.1 20250110 (Red Hat 14.2.1-7)	 64 bit
Svt[info]: LIB Build date: Jul  3 2025 03:14:07
Svt[info]: -------------------------------------------
Svt[info]: Level of Parallelism: 5
Svt[info]: Number of PPCS 140
Svt[info]: [asm level on system : up to avx2]
Svt[info]: [asm level selected : up to avx2]
Svt[info]: -------------------------------------------
Svt[info]: SVT [config]: main profile	tier (auto)	level (auto)
Svt[info]: SVT [config]: width / height / fps numerator / fps denominator 		: 640 / 480 / 30 / 1
Svt[info]: SVT [config]: bit-depth / color format 					: 8 / YUV420
Svt[info]: SVT [config]: preset / tune / pred struct 					: 8 / PSNR / random access
Svt[info]: SVT [config]: gop size / mini-gop size / key-f

LeRobotDataset({
    Repository ID: 'jonathm126/eval_so101_leg_pick_and_place_50_episodes_v2_real_v1',
    Number of selected episodes: '2',
    Number of selected samples: '3464',
    Features: '['action', 'observation.state', 'observation.images.wrist_cam', 'observation.images.top_cam', 'timestamp', 'frame_index', 'episode_index', 'index', 'task_index']',
})',